# 01 · Pipeline walkthrough

How raw USAspending responses become the analytical tables the findings rest on, and the evidence that the transformation is correct.

**Stages:** `ingest → analyse → warehouse → validate → report`

Every stage reads the previous stage from disk, so any stage can be re-run alone. See [`docs/METHODOLOGY.md`](../docs/METHODOLOGY.md) for the reasoning behind each analytical choice.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.width", 190)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

# Categorical slots validated for colour-vision-deficiency separation and
# contrast against a light surface (see the project's dataviz palette).
BLUE, ORANGE, AQUA, RED = "#2a78d6", "#eb6834", "#1baf7a", "#e34948"
plt.rcParams.update({
    "figure.figsize": (10, 4.4), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.7,
    "axes.titlesize": 12, "axes.titleweight": "semibold",
    "font.size": 10, "axes.labelsize": 10,
})

CURATED = ROOT / "data" / "curated"
SAMPLES = ROOT / "data" / "samples"

def load(name):
    """Read a curated table, falling back to the committed sample CSV."""
    parquet = CURATED / f"{name}.parquet"
    if parquet.exists():
        return pd.read_parquet(parquet)
    return pd.read_csv(SAMPLES / f"{name}.csv")

print("pandas", pd.__version__, "| numpy", np.__version__)


pandas 2.0.3 | numpy 1.26.4


## 1. Configuration is data, not code

Every judgement call — which FPDS codes count as competed, how pricing types map to risk buckets, the materiality floor, the index weights — lives in `config/pipeline.yml`. A reviewer can change an assumption and re-run without reading the implementation.

In [2]:
from fedspend.config import Config

cfg = Config.load(ROOT / 'config' / 'pipeline.yml')
print('fiscal years      :', cfg.fiscal_years)
print('award types       :', cfg.award_type_codes)
print('competed codes    :', cfg.competed_codes)
print('not-competed codes:', cfg.not_competed_codes)
print('materiality floor : ${:,.0f}'.format(cfg['analysis']['materiality_floor_usd']))
print()
print('index weights:')
for dim, w in cfg['efficiency_index']['weights'].items():
    print(f'  {dim:<18} {w:.2f}')

fiscal years      : [2021, 2022, 2023, 2024, 2025]
award types       : ['A', 'B', 'C', 'D']
competed codes    : ['A', 'D', 'F', 'CDO']
not-competed codes: ['B', 'C', 'E', 'G', 'NDO']
materiality floor : $1,000,000,000

index weights:
  competition        0.30
  pricing_risk       0.25
  vendor_diversity   0.20
  spend_discipline   0.15
  stability          0.10


### The `E` decision

FPDS code `E` is *follow-on to competed action*. The parent award was competed; this action was not. Agencies differ on the convention, so the project follows the FPDS-NG treatment (E = not competed) **and** re-runs the whole index under the opposite convention in the sensitivity analysis. It is a config value, not a constant buried in a function.

In [3]:
buckets = pd.Series(cfg.pricing_bucket_by_code).rename('risk_bucket')
labels = {str(c): l for b in cfg['pricing_risk'].values() for c, l in b.items()}
pd.DataFrame({'label': pd.Series(labels), 'risk_bucket': buckets}).sort_values('risk_bucket')

,label,risk_bucket
R,Cost plus award fee,cost_reimbursement
S,Cost no fee,cost_reimbursement
T,Cost sharing,cost_reimbursement
U,Cost plus fixed fee,cost_reimbursement
V,Cost plus incentive fee,cost_reimbursement
A,Fixed price redetermination,fixed_price
B,Fixed price level of effort,fixed_price
J,Firm fixed price,fixed_price
K,Fixed price with economic price adjustment,fixed_price
L,Fixed price incentive,fixed_price


## 2. Inflation adjustment

The GDP implicit price deflator, not CPI-U: the subject is government purchases of goods and services, not a household basket. A fiscal-year deflator is the mean of the four calendar quarters beginning October of the prior year — exactly the federal fiscal year, with no interpolation.

In [4]:
deflator = pd.read_parquet(CURATED.parent / 'interim' / 'deflator_fy.parquet') \
    if (CURATED.parent / 'interim' / 'deflator_fy.parquet').exists() else None
if deflator is None:
    gov = load('gov_summary')
    deflator = gov[['fiscal_year', 'real_multiplier']]
deflator

,fiscal_year,deflator,quarters_observed,deflator_base_fy,real_multiplier,cumulative_inflation_vs_base_pct
0,2021,108.502,4,2025,1.179,-15.184
1,2022,116.183,4,2025,1.101,-9.180
2,2023,121.572,4,2025,1.052,-4.968
3,2024,124.660,4,2025,1.026,-2.553
4,2025,127.926,4,2025,1.000,0.000


In [5]:
gov = load('gov_summary').sort_values('fiscal_year')
cum = (gov['real_multiplier'].iloc[0] - 1) * 100
print(f'Cumulative FY{gov.fiscal_year.iloc[0]}-FY{gov.fiscal_year.iloc[-1]} inflation: {cum:.1f}%')
print('A nominal increase smaller than this is a real DECREASE.')

Cumulative FY2021-FY2025 inflation: 17.9%
A nominal increase smaller than this is a real DECREASE.


## 3. Reconciliation — the check that makes the rest trustworthy

The competition table is assembled from **nine separate API calls per agency-year**, one per FPDS extent-competed code. The pricing table comes from sixteen. If those slices do not sum back to the independently retrieved agency total, the extraction is wrong and every downstream figure is suspect.

This is the single most important cell in the project.

In [6]:
interim = CURATED.parent / 'interim'
if (interim / 'agency_fy.parquet').exists():
    agency = pd.read_parquet(interim / 'agency_fy.parquet')
    comp = pd.read_parquet(interim / 'agency_fy_competition.parquet')
    sliced = comp.groupby(['fiscal_year','agency_name'], as_index=False)['obligations'].sum()
    m = agency.merge(sliced, on=['fiscal_year','agency_name'], suffixes=('_total','_sliced'))
    m = m[m['obligations_total'].abs() > 1e6]
    m['rel_diff'] = (m.obligations_sliced - m.obligations_total) / m.obligations_total
    print(f'agency-years checked      : {len(m)}')
    print(f'max |relative difference| : {m.rel_diff.abs().max():.2e}')
    print(f'breaches of 1% tolerance  : {(m.rel_diff.abs() > 0.01).sum()}')
else:
    print('interim extracts not present - run `make ingest` to reproduce this check')

agency-years checked      : 325
max |relative difference| : 3.46e-03
breaches of 1% tolerance  : 0


The residual difference comes from transactions USAspending assigns a null extent-competed code, which no code-filtered query returns. That is why the contract uses a 1% tolerance rather than demanding exact equality — and why the tolerance is stated rather than tuned until it passed.

## 4. The full validation suite

In [7]:
report = ROOT / 'outputs' / 'tables' / 'validation_report.csv'
if report.exists():
    display(pd.read_csv(report)[['name','passed','severity','message']])
else:
    print('run `make validate` to generate the report')

,name,passed,severity,message
0,fiscal_year_coverage,True,ERROR,"Expected [2021, 2022, 2023, 2024, 2025], found..."
1,unique_agency_year_key,True,ERROR,"0 duplicate (fiscal_year, agency_name) rows"
2,competition_slices_reconcile_to_totals,True,ERROR,0 of 325 agency-years exceed 1.0% reconciliati...
3,pricing_slices_reconcile_to_totals,True,ERROR,0 of 325 agency-years exceed 1.0% tolerance
4,monthly_slices_reconcile_to_annual,True,ERROR,0 of 5 fiscal years exceed 1.0% tolerance
5,shares_within_unit_interval,True,ERROR,"all shares in [0,1]"
6,deflator_monotonic_increasing,True,WARN,GDP deflator rises across the window
7,negative_net_obligations,False,WARN,4 agency-years have net-negative obligations (...
8,index_population_above_materiality_floor,True,ERROR,0 scored agencies fall below the $1.0B floor
9,shift_share_reconciles_exactly,True,ERROR,max |residual| = 3.43e-16 (tolerance 1e-09)


## 5. What comes out

21 curated tables. `fact_agency_year` is the centre of the model: one row per agency per fiscal year carrying every derived measure.

In [8]:
fact = load('fact_agency_year_material') if not (CURATED/'fact_agency_year.parquet').exists() \
       else load('fact_agency_year')
print(f'{len(fact)} rows x {fact.shape[1]} columns')
print()
for c in fact.columns:
    print(' ', c)

345 rows x 45 columns

  fiscal_year
  agency_id
  agency_code
  agency_name
  agency_slug
  obligations
  real_multiplier
  obligations_real
  competed_obligations
  not_competed_obligations
  competed_share
  fixed_price_share
  cost_reimbursement_share
  labor_hour_share
  government_risk_share
  government_risk_obligations
  setaside_obligations
  setaside_share
  september_obligations
  september_share
  q4_share
  surge_ratio
  september_excess_obligations
  vendor_hhi
  hhi_normalized
  vendor_hhi_class
  vendor_gini
  vendor_cr1
  vendor_cr4
  vendor_cr10
  vendor_coverage_share
  vendors_observed
  prior_year
  change_abs
  change_pct
  prior_year_real
  change_abs_real
  change_pct_real
  real_growth_robust_z
  real_growth_iqr_upper
  real_growth_iqr_lower
  real_growth_is_high_outlier
  real_growth_is_low_outlier
  real_growth_is_outlier
  is_material


In [9]:
cols = ['fiscal_year','agency_name','obligations','obligations_real',
        'competed_share','government_risk_share','september_share','vendor_hhi']
fact[fact.fiscal_year == fact.fiscal_year.max()].nlargest(10, 'obligations')[cols]

,fiscal_year,agency_name,obligations,obligations_real,competed_share,government_risk_share,september_share,vendor_hhi
276,2025,Department of Defense,"491,685,487,925.190","491,685,487,925.190",0.527,0.304,0.188,99.448
277,2025,Department of Veterans Affairs,"78,313,337,208.430","78,313,337,208.430",0.922,0.016,0.129,"1,234.202"
278,2025,Department of Energy,"48,549,337,828.870","48,549,337,828.870",0.970,0.968,0.088,515.200
279,2025,Department of Homeland Security,"28,336,601,669.430","28,336,601,669.430",0.910,0.198,0.426,184.730
280,2025,General Services Administration,"23,810,677,070.150","23,810,677,070.150",0.921,0.550,0.178,225.385
281,2025,Department of Health and Human Services,"21,290,230,376.020","21,290,230,376.020",0.885,0.492,0.284,79.634
282,2025,National Aeronautics and Space Administration,"15,782,680,383.900","15,782,680,383.900",0.660,0.668,0.209,449.900
283,2025,Department of State,"9,865,475,459.760","9,865,475,459.760",0.734,0.337,0.309,131.040
284,2025,Department of Agriculture,"9,569,572,358.200","9,569,572,358.200",0.891,0.018,0.209,39.570
285,2025,Department of Transportation,"9,022,525,885.220","9,022,525,885.220",0.815,0.358,0.211,161.076
